# 03 - Clusterização de Regimes de Mercado
## Identificação de regimes usando K-Means sobre indicadores agregados
O objetivo é classificar cada dia de negociação em um regime de mercado
(ex: alta com baixa volatilidade, queda com alta volatilidade, lateral).
O rótulo do regime será usado como feature adicional no modelo supervisionado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("../data/processed/features.parquet")
print(f"Shape: {df.shape}")
print(f"Período: {df.index.get_level_values('date').min()} até {df.index.get_level_values('date').max()}")

### 3.1 Agregação diária das features
Para identificar o regime do mercado como um todo, calculamos a mediana
de cada feature entre todas as ações em cada dia. A mediana é mais robusta
que a média porque não é distorcida por valores extremos de ações individuais.

In [ ]:
regime_features = [
    "ret_1d", "ret_5d", "ret_21d",
    "volatility_gk_21d", "volatility_std_21d",
    "rsi_14",
    "bb_width",
    "volume_ratio_21d"
]

df_market = df.groupby(level="date")[regime_features].median()

print(f"Shape do dataframe agregado: {df_market.shape}")
df_market.head()

### 3.2 Normalização
O K-Means é sensível à escala das variáveis. Uma feature que varia entre 0 e 100
(como o RSI) dominaria o cálculo de distância sobre uma que varia entre 0 e 0.05
(como retorno diário). O StandardScaler coloca todas na mesma escala
(média 0, desvio padrão 1).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_market)

print(f"Shape normalizado: {X_scaled.shape}")
print(f"Média por feature (deve ser ~0): {X_scaled.mean(axis=0).round(4)}")
print(f"Desvio padrão por feature (deve ser ~1): {X_scaled.std(axis=0).round(4)}")

### 3.3 Escolha do número de clusters
Testamos de 2 a 8 clusters e avaliamos com duas métricas:
- Inércia: soma das distâncias ao quadrado de cada ponto ao seu centróide. Menor é melhor.
- Silhouette Score: mede quão bem cada ponto pertence ao seu cluster vs os outros. Maior é melhor (máximo 1).

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, "bo-")
ax1.set_xlabel("Número de clusters (K)")
ax1.set_ylabel("Inércia")
ax1.set_title("Método do cotovelo")

ax2.plot(K_range, silhouettes, "ro-")
ax2.set_xlabel("Número de clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score por K")

plt.tight_layout()
plt.savefig("../results/regime_cluster_selection.png", dpi=150, bbox_inches="tight")
plt.show()

for k, s in zip(K_range, silhouettes):
    print(f"K={k}: Silhouette={s:.4f}")

### 3.4 Ajuste do modelo final
Com base nos gráficos acima, escolhemos K=4.
Quatro regimes capturam os cenários principais sem fragmentar demais os dados.

In [ ]:
K_FINAL = 4

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df_market["regime"] = km_final.fit_predict(X_scaled)

print("Distribuição dos regimes:")
print(df_market["regime"].value_counts().sort_index())
print(f"\nSilhouette Score: {silhouette_score(X_scaled, df_market['regime']):.4f}")

### 3.5 Interpretação dos regimes
Analisamos as características médias de cada regime para entendê-los.

In [ ]:
regime_profiles = df_market.groupby("regime")[regime_features].mean()
print("Perfil médio de cada regime:")
print(regime_profiles.round(6).T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_features = ["ret_21d", "volatility_std_21d", "rsi_14", "bb_width"]

for ax, feat in zip(axes.flatten(), plot_features):
    for regime in sorted(df_market["regime"].unique()):
        data = df_market[df_market["regime"] == regime][feat]
        ax.hist(data, bins=30, alpha=0.5, label=f"Regime {regime}", density=True)
    ax.set_title(feat)
    ax.legend()

plt.suptitle("Distribuição das features por regime", fontsize=14)
plt.tight_layout()
plt.savefig("../results/regime_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.6 Nomeação dos regimes
Com base nos perfis, atribuímos nomes descritivos a cada cluster.
Os nomes são definidos após inspecionar a tabela acima.

In [ ]:
regime_stats = df_market.groupby("regime")[["ret_21d", "volatility_std_21d"]].mean()
regime_stats = regime_stats.sort_values("ret_21d")

regime_names = {}
sorted_regimes = regime_stats.index.tolist()
labels = ["crise", "baixa", "neutro", "alta"]

for regime_id, label in zip(sorted_regimes, labels):
    regime_names[regime_id] = label
    ret = regime_stats.loc[regime_id, "ret_21d"]
    vol = regime_stats.loc[regime_id, "volatility_std_21d"]
    print(f"Regime {regime_id} -> '{label}' (ret_21d={ret:.6f}, vol={vol:.6f})")

df_market["regime_name"] = df_market["regime"].map(regime_names)

print("\nDistribuição por nome:")
print(df_market["regime_name"].value_counts())

### 3.7 Visualização temporal dos regimes

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

colors = {"crise": "red", "baixa": "orange", "neutro": "gray", "alta": "green"}

for regime_name, color in colors.items():
    mask = df_market["regime_name"] == regime_name
    dates = df_market.index[mask]
    ax1.scatter(dates, df_market.loc[mask, "ret_21d"], c=color, s=2, label=regime_name, alpha=0.6)

ax1.set_ylabel("Retorno 21d (mediana do mercado)")
ax1.legend(loc="upper right")
ax1.set_title("Regimes de mercado ao longo do tempo")
ax1.axhline(y=0, color="black", linewidth=0.5)

ax2.plot(df_market.index, df_market["volatility_std_21d"], color="navy", linewidth=0.5)
for regime_name, color in colors.items():
    mask = df_market["regime_name"] == regime_name
    ax2.fill_between(df_market.index, 0, df_market["volatility_std_21d"],
                     where=mask, color=color, alpha=0.3)
ax2.set_ylabel("Volatilidade 21d")
ax2.set_xlabel("Data")

plt.tight_layout()
plt.savefig("../results/regime_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.8 Atribuir regime aos dados individuais e salvar
Cada ação recebe o regime do dia correspondente.
Salvamos o dataset atualizado.

In [ ]:
regime_map = df_market[["regime", "regime_name"]].copy()
regime_map.columns = ["market_regime", "market_regime_name"]

df = df.join(regime_map, on="date", how="left")

print(f"Shape final: {df.shape}")
print(f"Colunas adicionadas: market_regime, market_regime_name")
print(f"\nDistribuição dos regimes no dataset completo:")
print(df["market_regime_name"].value_counts())

df.to_parquet("../data/processed/features_with_regime.parquet")
print("\nSalvo em data/processed/features_with_regime.parquet")